# headswap_V2 — Krea 2 Identity Edit

**Deterministic head / face swap** using [Krea 2 Turbo](https://huggingface.co/Comfy-Org/Krea-2) + the community [Identity Edit](https://huggingface.co/conradlocke/krea2-identity-edit) LoRA.

This notebook is a **demo runner** for the `headswap_V2` repository. It clones the repo, caches weights on Google Drive, uploads two photos, and runs the existing pipeline — it does not reimplement the model.

| Input | Role |
| --- | --- |
| **Body** (image 1) | Scene — pose, clothing, background kept |
| **Face** (image 2) | Identity — face / head to transfer |

**Recommended runtime:** GPU → **A100** (40 GB). T4 works but is much slower (~3–4 min/image).

> **How to use:** Runtime → Change runtime type → GPU (A100 if available) → Run all.


---
## 1 · Check the GPU

Confirm CUDA is available before downloading ~18 GB of weights.


In [ ]:
import sys
from pathlib import Path

# Detect Colab early
IN_COLAB = Path("/content").exists()
assert IN_COLAB, "This demo notebook is intended for Google Colab (/content)."

import torch
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → GPU "
    "(prefer A100), then Runtime → Run all."
)
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
print(f"Torch: {torch.__version__}")


---
## 2 · Mount Google Drive (model cache)

Weights are stored under  
`/content/drive/MyDrive/headswap_V2/models`  
so reconnecting the runtime does **not** require a full re-download.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted at /content/drive")


---
## 3 · Clone the repository

Pulls the latest `main` from GitHub into `/content/headswap_V2`.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("Repository:", Path.cwd())


---
## 4 · Configure environment

Sets ComfyUI + persistent model paths and prints a short summary.


In [ ]:
import importlib.util
from pathlib import Path

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location(
    "colab_env", REPO / "scripts" / "colab_env.py"
)
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)

PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))
colab_env.print_banner(PATHS)


---
## 5 · Install dependencies & ComfyUI

Idempotent setup: ComfyUI, Python requirements, aria2, and the `comfyui-krea2edit` nodes.
Model weights are downloaded after Hugging Face login so the cache can authenticate cleanly.


In [ ]:
import os
from pathlib import Path

os.chdir("/content/headswap_V2")
print("cwd:", Path.cwd())

# ComfyUI + Python deps + Krea2 custom nodes (weights after HF login)
!bash scripts/setup_colab.sh
!bash scripts/setup_krea2_nodes.sh


---
## 6 · Hugging Face login

Some assets are gated or rate-limited when anonymous. Paste a token with **read** access.


In [ ]:
from huggingface_hub import login

login()  # paste token when prompted
print("Hugging Face login OK")


---
## 7 · Download / verify models

~18 GB total (Turbo FP8 UNet, Qwen3-VL, VAE, Identity Edit LoRA).  
If Drive already has complete files, this step only verifies and re-links into ComfyUI.


In [ ]:
import os
from pathlib import Path

os.chdir("/content/headswap_V2")
store = Path(os.environ["HEADSWAP_MODEL_STORE"])
print("Model store:", store)

!python scripts/download_krea2.py \
  --comfy "$COMFYUI_PATH" \
  --store-dir "$HEADSWAP_MODEL_STORE" \
  --staging-dir "$HEADSWAP_STAGING_DIR" \
  --backend auto \
  --disable-xet

required = [
    store / "diffusion_models" / "krea2_turbo_fp8_scaled.safetensors",
    store / "text_encoders" / "qwen3vl_4b_fp8_scaled.safetensors",
    store / "vae" / "qwen_image_vae.safetensors",
    store / "loras" / "krea2_identity_edit_v1_2_r64.safetensors",
]
missing = [p for p in required if not p.exists()]
assert not missing, f"Missing weights: {missing}"
print("All required Krea2 weights present.")


---
## 8 · Upload body & face images

1. Run the cell  
2. Upload **body** (scene person) when prompted  
3. Upload **face** (identity) when prompted  

Files are saved as `data/custom/body.png` and `data/custom/face.png`.


In [ ]:
from pathlib import Path
from google.colab import files
from PIL import Image
from IPython.display import display, Markdown

custom = Path("data/custom")
custom.mkdir(parents=True, exist_ok=True)

print("Upload BODY image (scene / clothing / pose)…")
up_body = files.upload()
assert up_body, "No body image uploaded."
body_name = next(iter(up_body))
body_path = custom / "body.png"
Image.open(__import_("io").BytesIO(up_body[body_name])).convert("RGB").save(body_path)

print("Upload FACE image (identity donor)…")
up_face = files.upload()
assert up_face, "No face image uploaded."
face_name = next(iter(up_face))
face_path = custom / "face.png"
Image.open(__import_("io").BytesIO(up_face[face_name])).convert("RGB").save(face_path)

display(Markdown("### Inputs"))
print("Body (scene):")
display(Image.open(body_path).resize((384, 384)))
print("Face (identity):")
display(Image.open(face_path).resize((384, 384)))
print(f"Saved → {body_path.resolve()} , {face_path.resolve()}")


---
## 9 · Prepare the evaluation pair

Registers your uploads as eval pair `custom_001` for the harness.


In [ ]:
!python scripts/prepare_eval_set.py --custom data/custom
print("Eval pair ready: custom_001")


---
## 10 · Run inference

Runs `configs/krea2_identity_edit.yaml` through the repository pipeline  
(`scripts/run_pipeline.py`). On A100 expect roughly **under ~2 minutes** cold;  
T4 is typically ~3–4 minutes.


In [ ]:
import importlib.util
import json
import time
from pathlib import Path

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.apply_env(colab_env.default_paths(use_drive=True))

from headswap.eval.runner import run_eval

CONFIG = Path("configs/krea2_identity_edit.yaml")
OUT = Path("results/krea2_identity_edit")

t0 = time.time()
report = run_eval(
    CONFIG,
    out_dir=OUT,
    force_mock=False,
    limit=1,
    pair_ids=["custom_001"],
)
wall = time.time() - t0

row = next(r for r in report["pairs"] if r.get("pair_id") == "custom_001")
assert row.get("success"), (
    f"Run failed: {row.get('fail_reasons') or (row.get('meta') or {}).get('run_error')}"
)

RESULT = Path(row["result_path"] or OUT / "images" / "custom_001" / "result.png")
assert RESULT.exists(), RESULT

OUT_COPY = Path("/content/headswap_outputs")
OUT_COPY.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%d_%H%M%S")
DEMO = OUT_COPY / f"krea2_result_{stamp}.png"
DEMO.write_bytes(RESULT.read_bytes())

print(f"Wall time:   {wall:.1f}s")
print(f"Latency:     {float(row.get('latency_s')):.1f}s")
print(f"Result:      {RESULT}")
print(f"Demo copy:   {DEMO}")


---
## 11 · Results

Side-by-side preview, stage timings, and a one-click download.


In [ ]:
import json
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown
from google.colab import files

# Reload latest successful row
metrics = Path("results/krea2_identity_edit/metrics.json")
report = json.loads(metrics.read_text())
row = next(r for r in report["pairs"] if r.get("pair_id") == "custom_001")
result = Path(row["result_path"])
body = Path("data/custom/body.png")
face = Path("data/custom/face.png")

display(Markdown("### Body · Face · Result"))
imgs = [
    Image.open(body).convert("RGB"),
    Image.open(face).convert("RGB"),
    Image.open(result).convert("RGB"),
]
# Uniform preview height
h = 320
resized = []
for im in imgs:
    w = int(im.width * (h / im.height))
    resized.append(im.resize((w, h)))
gap = 16
canvas_w = sum(im.width for im in resized) + gap * (len(resized) - 1)
canvas = Image.new("RGB", (canvas_w, h), (255, 255, 255))
x = 0
for im in resized:
    canvas.paste(im, (x, 0))
    x += im.width + gap
display(canvas)

timing = (row.get("meta") or {}).get("timing_s") or {}
display(Markdown("### Latency breakdown"))
print(f"total latency: {row.get('latency_s'):.2f}s")
if timing:
    for k, v in timing.items():
        print(f"  {k:<28} {float(v):7.2f}s")
else:
    print("  (no stage timings in meta)")

# Prefer the stamped demo copy if present
candidates = sorted(Path("/content/headswap_outputs").glob("krea2_result_*.png"))
download_path = candidates[-1] if candidates else result
display(Markdown("### Download"))
print(download_path)
files.download(str(download_path))


---
## Notes

- **Architecture:** notebook → `run_eval` → `Krea2IdentityEditPipeline` → ComfyUI + `comfyui-krea2edit`. Pipeline logic lives in the repo, not in this notebook.
- **Caching:** weights on Drive; ComfyUI under `/content/ComfyUI` (rebuilt if the runtime is reset).
- **Quality knobs:** edit `configs/krea2_identity_edit.yaml` (`steps`, `ref_boost`, `grounding_px`, `seed`) then re-run §10.
- **Support:** see `COLAB.md` in the repository root.

*Magic Hour · headswap_V2 demo*
